# 02 - ImageNet-100 Healer VAE

Train a Variational Autoencoder (VAE) to "heal" corrupted ImageNet-100 images:
- Input: Noisy RGB images (224x224x3)
- Output: Clean reconstructed images
- Architecture: 5-block encoder/decoder for 224x224 resolution

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np
import os

# Configuration
BASE_DIR = r"C:\Users\Rushikesh\OneDrive\CODES\SelfHealingNN"
os.chdir(BASE_DIR)

torch.set_num_threads(8)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

IMAGE_SIZE = 224
BATCH_SIZE = 32
LATENT_DIM = 256
NUM_WORKERS = 4

print(f"Device: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 1. Load Data

In [ ]:
# Data paths
DATA_DIR = os.path.join(BASE_DIR, "imagenet100_data")
TRAIN_DIR = os.path.join(DATA_DIR, "train")
VAL_DIR = os.path.join(DATA_DIR, "val")

# Transforms (no normalization for VAE)
train_transforms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1),
    transforms.ToTensor(),
])

val_transforms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
])

# Datasets
train_dataset = datasets.ImageFolder(root=TRAIN_DIR, transform=train_transforms)
val_dataset = datasets.ImageFolder(root=VAL_DIR, transform=val_transforms)

# Dataloaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, 
                          num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=True)

print(f"Train: {len(train_dataset):,} images ({len(train_loader)} batches)")
print(f"Val: {len(val_dataset):,} images ({len(val_loader)} batches)")
print(f"Classes: {len(train_dataset.classes)}")

## 2. Noise Function

In [ ]:
def add_noise(image_tensor, noise_factor=0.3):
    """Add Gaussian noise to RGB images."""
    noise = torch.randn_like(image_tensor) * noise_factor
    return torch.clamp(image_tensor + noise, 0., 1.)

print("Noise function ready")

## 3. VAE Architecture

5-block encoder/decoder for 224x224 RGB images:
- Encoder: 224 -> 112 -> 56 -> 28 -> 14 -> 7
- Decoder: 7 -> 14 -> 28 -> 56 -> 112 -> 224

In [ ]:
class ImageNet100VAE(nn.Module):
    """
    VAE for 224x224 RGB images (ImageNet-100).
    
    Architecture:
    - Encoder: 5 conv blocks with stride 2 (224->112->56->28->14->7)
    - Latent: 256-dimensional
    - Decoder: 5 transposed conv blocks (7->14->28->56->112->224)
    
    Parameters: ~7.5M
    """
    
    def __init__(self, latent_dim=256):
        super(ImageNet100VAE, self).__init__()
        self.latent_dim = latent_dim
        
        # Encoder: 224 -> 112 -> 56 -> 28 -> 14 -> 7
        self.encoder = nn.Sequential(
            # Block 1: 224 -> 112
            nn.Conv2d(3, 32, 3, stride=2, padding=1),
            nn.BatchNorm2d(32),
            nn.LeakyReLU(0.2),
            
            # Block 2: 112 -> 56
            nn.Conv2d(32, 64, 3, stride=2, padding=1),
            nn.BatchNorm2d(64),
            nn.LeakyReLU(0.2),
            
            # Block 3: 56 -> 28
            nn.Conv2d(64, 128, 3, stride=2, padding=1),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2),
            
            # Block 4: 28 -> 14
            nn.Conv2d(128, 256, 3, stride=2, padding=1),
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.2),
            
            # Block 5: 14 -> 7
            nn.Conv2d(256, 512, 3, stride=2, padding=1),
            nn.BatchNorm2d(512),
            nn.LeakyReLU(0.2),
        )
        
        # Flattened size: 512 * 7 * 7 = 25,088
        self.flatten_size = 512 * 7 * 7
        
        # Latent space
        self.fc_mu = nn.Linear(self.flatten_size, latent_dim)
        self.fc_logvar = nn.Linear(self.flatten_size, latent_dim)
        self.fc_decode = nn.Linear(latent_dim, self.flatten_size)
        
        # Decoder: 7 -> 14 -> 28 -> 56 -> 112 -> 224
        self.decoder = nn.Sequential(
            # Block 1: 7 -> 14
            nn.ConvTranspose2d(512, 256, 3, stride=2, padding=1, output_padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            
            # Block 2: 14 -> 28
            nn.ConvTranspose2d(256, 128, 3, stride=2, padding=1, output_padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            
            # Block 3: 28 -> 56
            nn.ConvTranspose2d(128, 64, 3, stride=2, padding=1, output_padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            
            # Block 4: 56 -> 112
            nn.ConvTranspose2d(64, 32, 3, stride=2, padding=1, output_padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            
            # Block 5: 112 -> 224
            nn.ConvTranspose2d(32, 3, 3, stride=2, padding=1, output_padding=1),
            nn.Sigmoid(),  # Output in [0, 1]
        )
    
    def reparameterize(self, mu, logvar):
        """Reparameterization trick."""
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std
    
    def forward(self, x):
        # Encode
        h = self.encoder(x)
        h = h.view(-1, self.flatten_size)
        
        # Latent
        mu = self.fc_mu(h)
        logvar = self.fc_logvar(h)
        z = self.reparameterize(mu, logvar)
        
        # Decode
        h2 = F.relu(self.fc_decode(z))
        h2 = h2.view(-1, 512, 7, 7)
        recon = self.decoder(h2)
        
        return recon, mu, logvar


# Create model and count parameters
model = ImageNet100VAE(latent_dim=LATENT_DIM).to(device)
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"ImageNet100VAE Architecture:")
print(f"  Input: (B, 3, 224, 224)")
print(f"  Latent dim: {LATENT_DIM}")
print(f"  Output: (B, 3, 224, 224)")
print(f"  Total parameters: {total_params:,}")
print(f"  Trainable parameters: {trainable_params:,}")

## 4. Loss Function

In [ ]:
def vae_loss(recon_x, x, mu, logvar, beta=0.5):
    """
    VAE loss = Reconstruction loss + beta * KL divergence
    
    Args:
        recon_x: Reconstructed images
        x: Original clean images
        mu: Mean of latent distribution
        logvar: Log variance of latent distribution
        beta: Weight for KL divergence (beta-VAE)
    """
    # Reconstruction loss (BCE for [0,1] images)
    BCE = F.binary_cross_entropy(recon_x, x, reduction='sum')
    
    # KL divergence
    KLD = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    
    return BCE + beta * KLD

print("Loss function ready (BCE + beta*KLD, beta=0.5)")

## 5. Training

In [ ]:
# Training configuration
EPOCHS = 30
LEARNING_RATE = 1e-3
NOISE_FACTOR = 0.3
BETA = 0.5

# Create models directory
os.makedirs(os.path.join(BASE_DIR, "models"), exist_ok=True)

# Optimizer and scheduler
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-5)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5, verbose=True)

print(f"Training configuration:")
print(f"  Epochs: {EPOCHS}")
print(f"  Learning rate: {LEARNING_RATE}")
print(f"  Noise factor: {NOISE_FACTOR}")
print(f"  Beta (KLD weight): {BETA}")
print(f"  Batches per epoch: {len(train_loader)}")

In [ ]:
# Training loop
print(f"\nStarting training on {device}...")
print("="*60)

train_losses = []
best_loss = float('inf')

for epoch in range(EPOCHS):
    model.train()
    epoch_loss = 0.0
    
    for batch_idx, (clean_images, _) in enumerate(train_loader):
        clean_images = clean_images.to(device)
        noisy_images = add_noise(clean_images, NOISE_FACTOR)
        
        # Forward pass
        optimizer.zero_grad()
        recon_images, mu, logvar = model(noisy_images)
        loss = vae_loss(recon_images, clean_images, mu, logvar, beta=BETA)
        
        # Backward pass
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        
        epoch_loss += loss.item()
        
        # Progress update
        if batch_idx % 100 == 0:
            avg_loss = epoch_loss / (batch_idx + 1)
            print(f"  Epoch {epoch+1}/{EPOCHS} | Batch {batch_idx:>4}/{len(train_loader)} | Loss: {avg_loss:,.1f}")
    
    # Epoch summary
    avg_epoch_loss = epoch_loss / len(train_loader)
    train_losses.append(avg_epoch_loss)
    scheduler.step(avg_epoch_loss)
    lr = optimizer.param_groups[0]['lr']
    
    print(f"Epoch {epoch+1}/{EPOCHS} | Avg Loss: {avg_epoch_loss:,.1f} | LR: {lr:.1e}")
    
    # Save best model
    if avg_epoch_loss < best_loss:
        best_loss = avg_epoch_loss
        torch.save(model.state_dict(), os.path.join(BASE_DIR, "models", "imagenet100_healer.pth"))
        print(f"  Saved best model (loss: {best_loss:,.1f})")
    
    print("-"*60)

print("\nTraining complete!")
print(f"Best loss: {best_loss:,.1f}")
print(f"Model saved to: models/imagenet100_healer.pth")

## 6. Training Curves

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(train_losses, 'b-', linewidth=2)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('VAE Training Loss')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Visualize Reconstruction Quality

In [ ]:
# Load best model
model.load_state_dict(torch.load(os.path.join(BASE_DIR, "models", "imagenet100_healer.pth")))
model.eval()

# Get sample images
images, labels = next(iter(val_loader))
images = images.to(device)
noisy = add_noise(images, 0.4)  # Test with higher noise

with torch.no_grad():
    healed, _, _ = model(noisy)

# Visualize
fig, axes = plt.subplots(3, 6, figsize=(18, 9))

titles = ["Original", "Corrupted", "Healed"]
rows = [images.cpu(), noisy.cpu(), healed.cpu()]

for row_idx, (row_imgs, title) in enumerate(zip(rows, titles)):
    for col in range(6):
        img = row_imgs[col].numpy().transpose(1, 2, 0)
        img = np.clip(img, 0, 1)
        axes[row_idx, col].imshow(img)
        axes[row_idx, col].axis('off')
        if col == 0:
            axes[row_idx, col].set_ylabel(title, fontsize=12, rotation=0, labelpad=60)

plt.suptitle("VAE Healing Quality (224x224 RGB) - Noise Factor 0.4", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Test different noise levels
noise_levels = [0.1, 0.2, 0.3, 0.4, 0.5]
sample_img = images[0:1]

fig, axes = plt.subplots(3, len(noise_levels), figsize=(15, 9))

for i, noise_level in enumerate(noise_levels):
    noisy = add_noise(sample_img, noise_level)
    with torch.no_grad():
        healed, _, _ = model(noisy)
    
    for row_idx, (img, title) in enumerate([
        (sample_img, "Original"),
        (noisy, f"Noisy ({noise_level})"),
        (healed, "Healed")
    ]):
        img_np = img[0].cpu().numpy().transpose(1, 2, 0)
        img_np = np.clip(img_np, 0, 1)
        axes[row_idx, i].imshow(img_np)
        axes[row_idx, i].axis('off')
        if i == 0:
            axes[row_idx, i].set_ylabel(title.split()[0], fontsize=10)
        if row_idx == 0:
            axes[row_idx, i].set_title(f"Noise: {noise_level}")

plt.suptitle("Healing Performance at Different Noise Levels", fontsize=14)
plt.tight_layout()
plt.show()

## 8. Summary

In [ ]:
print("="*50)
print("ImageNet-100 VAE Healer Training Complete!")
print("="*50)
print(f"Model: ImageNet100VAE")
print(f"Parameters: {total_params:,}")
print(f"Best loss: {best_loss:,.1f}")
print(f"")
print(f"Saved to: models/imagenet100_healer.pth")
print("="*50)
print("\nNext: Run 03_ImageNet100_Expert_ResNet.ipynb to train the classifier!")